In [ ]:
# ---- 1. setup ---------------------------------------------------------------------
# in the beginning everything was written to /content/outputs and the splits
# disappeared when the session ended, so now the outputs go to Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("drive mount failed:", e)   # this happens when the cell is re-run

import os

DATA = '/content/data'                         # the kaggle zip goes here
OUT  = '/content/drive/MyDrive/MDL_HW1'        # figures / tables / checkpoints
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

if not os.path.exists(DATA + '/HAM10000_metadata.csv'):
    print("downloading from Kaggle...")
    # TODO: move the token to a colab secret at some point. left here for now
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_9d722ed3fcd37402188b3ec9c53f0f99'
    !pip install -q kaggle --upgrade
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data --unzip
    print("dataset downloaded")
else:
    print("dataset already in /content/data")

!ls /content/data

In [ ]:
# ---- 2. dataset + split -----------------------------------------------------------
# HAM10000 has 10015 images but only 7470 lesions: the same lesion was
# photographed up to 4 times (different angles / body parts).
#
# v1 of this cell used train_test_split(stratify=dx) directly on the images,
# so the same lesion ended up in train and in test and the first results
# looked too good (0.83 acc for a CNN trained from scratch on 7k images is
# not believable). now the split is done on lesion_id and the images follow.
import pandas as pd
from sklearn.model_selection import train_test_split

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'mel', 'vasc']
SEED = 42

def make_grouped_split(meta, seed=SEED):
    """70/15/15 on lesion_id, stratified on the class of each lesion."""
    meta = meta.copy()
    meta['label']  = meta['dx'].map({c: i for i, c in enumerate(CLASSES)})
    meta['binary'] = (meta['dx'] == 'mel').astype(int)   # melanoma vs rest

    # one row per lesion, if a lesion has two dx values (rare) the first is used
    lesions = meta.groupby('lesion_id', as_index=False).first()[['lesion_id', 'label']]

    tr_l, tmp_l = train_test_split(lesions, test_size=0.30, stratify=lesions['label'], random_state=seed)
    va_l, te_l = train_test_split(tmp_l,  test_size=0.50, stratify=tmp_l['label'],     random_state=seed)

    tr = meta[meta['lesion_id'].isin(tr_l['lesion_id'])]
    va = meta[meta['lesion_id'].isin(va_l['lesion_id'])]
    te = meta[meta['lesion_id'].isin(te_l['lesion_id'])]
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

meta = pd.read_csv(os.path.join(DATA, 'HAM10000_metadata.csv'))
print("metadata:", meta.shape, " lesions:", meta['lesion_id'].nunique())

train_df, val_df, test_df = make_grouped_split(meta)

# sanity checks, printed on purpose (they go in the report, section 2.1)
overlap_tr_te = set(train_df['lesion_id']) & set(test_df['lesion_id'])
overlap_tr_va = set(train_df['lesion_id']) & set(val_df['lesion_id'])
print("overlap train/test lesions:", len(overlap_tr_te))
print("overlap train/val  lesions:", len(overlap_tr_va))
assert len(overlap_tr_te) == 0 and len(overlap_tr_va) == 0, "lesion leaked across splits!"

# pos_weight for the binary loss, computed from the train set only
pos = train_df['binary'].sum()
neg = len(train_df) - pos
POS_WEIGHT = float(neg) / float(pos)
print(f"pos_weight (neg/pos, train only) = {POS_WEIGHT:.3f}")

print("split sizes: train", len(train_df), " val", len(val_df), " test", len(test_df))
print()
print("images per class")
print("class    train  val  test")
for i, c in enumerate(CLASSES):
    a = int((train_df['label'] == i).sum())
    b = int((val_df['label'] == i).sum())
    d = int((test_df['label'] == i).sum())
    print(f"{c:<7} {a:5d} {b:4d} {d:5d}")

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(os.path.join(OUT, name + '.csv'), index=False)
print("splits saved to", OUT)

In [ ]:
# ---- 3. imports, seed, device -----------------------------------------------------
import os, random, time, json
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, confusion_matrix, roc_curve)

def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

MEAN = (0.485, 0.456, 0.406)     # imagenet statistics, the pretrained models expect them
STD  = (0.229, 0.224, 0.225)

In [ ]:
# ---- 4. dataset + dataloaders -----------------------------------------------------
# the kaggle zip has the images in two folders, we look in part_1 then part_2
IMG1 = os.path.join(DATA, 'HAM10000_images_part_1')
IMG2 = os.path.join(DATA, 'HAM10000_images_part_2')

def image_path(image_id):
    p1 = os.path.join(IMG1, image_id + '.jpg')
    return p1 if os.path.exists(p1) else os.path.join(IMG2, image_id + '.jpg')

# reload the splits that cell 1 wrote to Drive (the runtime may have restarted)
train_df = pd.read_csv(os.path.join(OUT, 'train.csv'))
val_df   = pd.read_csv(os.path.join(OUT, 'val.csv'))
test_df  = pd.read_csv(os.path.join(OUT, 'test.csv'))
for d in (train_df, val_df, test_df):
    d['path'] = d['image_id'].apply(image_path)

print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("missing image files:", (~train_df['path'].apply(os.path.exists)).sum())

In [ ]:
class HAM(Dataset):
    """returns (image tensor, label). label = 0..6 for multi, 0/1 for binary."""
    def __init__(self, df, transform, mode='multi'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        if self.mode == 'multi':
            y = int(row['label'])
        else:
            y = int(row['binary'])
        return img, y

def make_loaders(mode='multi', bs=32, use_sampler=True, size=224):
    """train loader is augmented + balanced, val/test are only resized."""
    train_tf = A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.ColorJitter(0.1, 0.1, 0.1, 0.05, p=0.5),
        # A.RandomBrightnessContrast(0.3, 0.3, p=1.0),  # too strong, washed out the dermoscopy
        A.Normalize(MEAN, STD),
        ToTensorV2(),
    ])
    eval_tf = A.Compose([A.Resize(height=size, width=size), A.Normalize(MEAN, STD), ToTensorV2()])

    tr_ds = HAM(train_df, train_tf, mode)
    va_ds = HAM(val_df,   eval_tf,  mode)
    te_ds = HAM(test_df,  eval_tf,  mode)

    if use_sampler:
        # the classes are very unbalanced (nv = 67%, df = 1%). without this the
        # model just answers 'nv' for everything and still gets 60% accuracy
        col = 'label' if mode == 'multi' else 'binary'
        counts = train_df[col].value_counts().sort_index().values
        w = 1.0 / counts
        sw = [w[k] for k in train_df[col].values]
        sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
        tr_loader = DataLoader(tr_ds, batch_size=bs, sampler=sampler, num_workers=2, pin_memory=True)
    else:
        tr_loader = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)

    va_loader = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, te_loader

trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)
xb, yb = next(iter(trl))
print("batch:", tuple(xb.shape), "labels:", tuple(yb.shape), "classes in batch:", sorted(yb.unique().tolist()))

In [ ]:
# ---- 5. custom CNN ----------------------------------------------------------------
# 5 stages, every stage = 2x conv3x3 + batchnorm, resolution halved between
# stages with maxpool, then global average pooling + one linear layer.
# 11.02M parameters (table 4 in the report).
class CustomCNN(nn.Module):
    def __init__(self, n_out=7, width=32, dropout=0.3):
        super().__init__()
        c1, c2, c3, c4, c5 = width, width*2, width*4, width*8, width*16
        self.features = nn.Sequential(
            # stage 1 -> 224
            nn.Conv2d(3, c1, 3, 1, 1, bias=False),  nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, 1, 1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            # stage 2 -> 112
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.Conv2d(c2, c2, 3, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            # stage 3 -> 56
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, 3, 1, 1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.Conv2d(c3, c3, 3, 1, 1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            # stage 4 -> 28
            nn.MaxPool2d(2),
            nn.Conv2d(c3, c4, 3, 1, 1, bias=False), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.Conv2d(c4, c4, 3, 1, 1, bias=False), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            # stage 5 -> 14
            nn.MaxPool2d(2),
            nn.Conv2d(c4, c5, 3, 1, 1, bias=False), nn.BatchNorm2d(c5), nn.ReLU(inplace=True),
            nn.Conv2d(c5, c5, 3, 1, 1, bias=False), nn.BatchNorm2d(c5), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(c5, n_out)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(self.drop(x))

_n = CustomCNN(7)
print("custom cnn params: %.2fM" % (sum(p.numel() for p in _n.parameters()) / 1e6))

In [ ]:
# ---- 6. evaluation function -------------------------------------------------------
def evaluate(model, loader, mode):
    """returns the metrics + the raw predictions (we need them for the tables)."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            if mode == 'multi':
                p = torch.softmax(out, dim=1).cpu().numpy()
            else:
                p = torch.sigmoid(out.view(-1)).cpu().numpy()
            ys.append(y.numpy())
            ps.append(p)
    y = np.concatenate(ys)
    p = np.concatenate(ps)

    if mode == 'multi':
        pred = p.argmax(1)
        acc = accuracy_score(y, pred)
        _, _, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
        return {'acc': acc, 'macro_f1': f1, 'y': y, 'p': p, 'pred': pred}
    else:
        pred = (p >= 0.5).astype(int)
        acc = accuracy_score(y, pred)
        auc = roc_auc_score(y, p)
        ap  = average_precision_score(y, p)
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
        return {'acc': acc, 'auc': auc, 'pr_auc': ap, 'precision': pr, 'recall': rc,
                'f1': f1, 'y': y, 'p': p, 'pred': pred}

print("evaluate ready")


def _print_class_report(res, title):
    """small helper, prints precision/recall/F1 per class for a multi model."""
    pr, rc, f1, sup = precision_recall_fscore_support(res['y'], res['pred'], labels=range(7), zero_division=0)
    print()
    print("per class -", title)
    print("class    prec   rec    f1    n")
    for i, c in enumerate(CLASSES):
        print(f"{c:<7} {pr[i]:.3f}  {rc[i]:.3f}  {f1[i]:.3f} {sup[i]:5d}")

In [ ]:
# ---- 8. train the custom cnn (7 classes) ------------------------------------------
# from scratch, 25 epochs max, early stop after 5 epochs without improvement.
# on a T4 one epoch is ~10 min with the sampler on the augmented train set.
EPOCHS = 25    # tried 30 first, it never got past ~18 because of early stop
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)

custom_multi = CustomCNN(n_out=7).to(device)
custom_multi, hist_custom_multi = train_model(custom_multi, trl, val_loader, 'multi',
                                             epochs=EPOCHS, lr=1e-3, tag='custom-7cls')

res = evaluate(custom_multi, te_loader, 'multi')
print()
print("TEST custom cnn (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'custom cnn / multi')

torch.save({'state_dict': custom_multi.state_dict(),
            'history': hist_custom_multi,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'custom_multi.pt'))
print("saved", os.path.join(OUT, 'custom_multi.pt'))

In [ ]:
# ---- 9. train the custom cnn (melanoma vs rest) -----------------------------------
# TODO: the sampler helps here too but it makes the loss very noisy, decide later
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=32)

custom_bin = CustomCNN(n_out=1).to(device)     # one logit -> BCEWithLogitsLoss
custom_bin, hist_custom_bin = train_model(custom_bin, trl_b, val_b, 'binary',
                                         epochs=EPOCHS, lr=1e-3, tag='custom-bin')

res_b = evaluate(custom_bin, te_b, 'binary')
print()
print("TEST custom cnn (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': custom_bin.state_dict(),
            'history': hist_custom_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'custom_binary.pt'))
print("saved", os.path.join(OUT, 'custom_binary.pt'))